# Strawberry Size Classification: Comparative Analysis of YOLO, Faster R-CNN, and EfficientDet
This notebook is designed to train and compare three different object detection models on a custom dataset of 427 strawberry images. The objective is to classify strawberries into three size categories: Short (0), Medium (1), and Long (2).

In [ ]:
# INITIAL SETUP AND IMPORTS

import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # For Confusion Matrix visualization

# Ultralytics for YOLO
!pip install ultralytics
from ultralytics import YOLO

# Mount Google Drive (if using Google Colab)
from google.colab import drive
drive.mount('/content/drive')

print(f"PyTorch Version: {torch.__version__}, Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 14.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Mounted at /content/drive
PyTorch Version: 2.11.0+cu130, Device: GPU


## 1. Data Preprocessing & Augmentation
Since the dataset is limited to 427 images, data augmentation is crucial to prevent overfitting and improve the models' generalization capabilities.

In [ ]:
# DATA AUGMENTATION PIPELINE
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# Define the target size
TARGET_SIZE = 640

transform = A.Compose([
    # 1. Resize the image so that its longest side is 640, without distorting the aspect ratio
    A.LongestMaxSize(max_size=TARGET_SIZE),

    # 2. Fill in the empty spaces with gray (114,114,114) to make a complete square
    A.PadIfNeeded(
        min_height=TARGET_SIZE,
        min_width=TARGET_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=[114, 114, 114], # Standard padding color for YOLO models
        position="center"
    ),

    # 3. Once the size and aspect ratio have been set, the remaining augmentation steps
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3),

    # 4. Divide the pixels by 255 to scale them to a range of 0–1 or to ImageNet standards
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),

    # 5. Tensor transformation
    ToTensorV2()

], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


/tmp/ipykernel_5282/445824487.py:14: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(


## 1.1 Split the dataset into Train (80%), Validation (10%), and Test (10%).

In [ ]:
import shutil
from sklearn.model_selection import train_test_split

!unzip -q /content/drive/MyDrive/dataset.zip -d /content/

SOURCE_IMG_DIR = "dataset/images"
SOURCE_LBL_DIR = "dataset/labels"

# Create the train, val, and test folders for YOLO in a single loop
splits = ["train", "val", "test"]
for split in splits:
    os.makedirs(f"yolo_dataset/images/{split}", exist_ok=True)
    os.makedirs(f"yolo_dataset/labels/{split}", exist_ok=True)

# Get filenames without extensions
all_files = [f.split('.')[0] for f in os.listdir(SOURCE_IMG_DIR) if f.endswith('.jpg')]

# Step 1: Split 80% of the data into the training set and the remaining 20% (validation + test)
train_files, temp_files = train_test_split(all_files, test_size=0.2, random_state=42)

# Step 2: Divide the remaining 20% exactly in half to create 10% Val and 10% Test
val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)

# A function that physically moves files
def move_files(file_list, split_type):
    for f in file_list:
        shutil.copy(f"{SOURCE_IMG_DIR}/{f}.jpg", f"yolo_dataset/images/{split_type}/{f}.jpg")

        txt_path = f"{SOURCE_LBL_DIR}/{f}.txt"
        if os.path.exists(txt_path):
            shutil.copy(txt_path, f"yolo_dataset/labels/{split_type}/{f}.txt")

# Start the copy process
move_files(train_files, "train")
move_files(val_files, "val")
move_files(test_files, "test")

print(f"The split is complete: {len(train_files)} Train, {len(val_files)} Val, {len(test_files)} Test")

The split is complete: 341 Train, 43 Val, 43 Test


## 2. Model 1: YOLO Training (You Only Look Once)
YOLO is a single-stage object detector known for its speed. It requires bounding box annotations in `.txt` format. We will train the YOLO model completely from scratch without using any pre-trained COCO weights. To do this, we initialize the model using the architecture configuration file (`.yaml`) instead of the pre-trained weights file (`.pt`).


In [ ]:
import yaml
from ultralytics import YOLO

# 1. Automatically Generating the data.yaml File
yaml_data = {
    'train': '/content/yolo_dataset/images/train',
    'val': '/content/yolo_dataset/images/val',
    'test': '/content/yolo_dataset/images/test',
    'nc': 3,
    'names': ['Short', 'Medium', 'Long'] # Class names
}

yaml_path = '/content/yolo_dataset/data.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"The data.yaml file has been created at {yaml_path}!")


# 2. YOLO Model Configuration and Training
# We initialize the model from scratch (with random weights).
yolo_model = YOLO('yolov8n.yaml')

print("Training is starting... This process may take a little while, depending on the size of your dataset.")

# Start Training
yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=150,     # 150 is a good starting point for beginning
    batch=16,
    imgsz=640,
    project='/content/drive/MyDrive/strawberry_project/results', # Save to Drive so the results are permanent
    name='train_run_scratch'
)

print("The training is complete! The weight files have been saved to Drive.")

The data.yaml file has been created at /content/yolo_dataset/data.yaml!
Training is starting... This process may take a little while, depending on the size of your dataset.
Ultralytics 8.4.182 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, lin

## 3. Model Faster R-CNN Training
Faster R-CNN is a two-stage object detector. We will initialize the complete architecture (including the ResNet50 backbone and the Region Proposal Network) with random weights by setting `weights=None` and `weights_backbone=None`.

## 4. Model EfficientDet Training
EfficientDet relies on a BiFPN architecture. We will initialize it randomly, ensuring no pre-trained backbone is loaded.

## 5. Validation Set Evaluation & Model Optimization
Before touching the test set, we evaluate our models on the Validation Set. This step is crucial to check for overfitting, analyze the training/validation loss curves, and determine if hyperparameter tuning (e.g., learning rate, augmentation limits) is required.

In [ ]:
# VALIDATION LOSS CURVES AND INTERMEDIATE mAP
import matplotlib.pyplot as plt

# TODO: Plot Training vs. Validation Loss curves for YOLO, Faster R-CNN, and EfficientDet.
# TODO: Calculate mAP on the Validation Set.

def plot_loss_curves(train_losses, val_losses, title="Model Loss Curve"):
    plt.figure(figsize=(8, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.show()

# If validation mAP is too low or validation loss increases (overfitting),
# we return to Step 2, 3, or 4 to adjust hyperparameters and retrain.

## 6. Inference on the Unseen Test Set
Once the models are optimized and finalized based on the validation results, we run inference on the strictly unseen Test Set. We will iterate through every test image and record the predictions (bounding boxes, classes, confidence scores) and inference times.

In [ ]:
# RUN INFERENCE ON ALL TEST IMAGES
import time
import glob
import os

test_images_path = '/content/drive/MyDrive/strawberry_project/test/images/*.jpg'
test_image_files = glob.glob(test_images_path)

# Initialize lists to store true labels and predicted labels for the final Confusion Matrix
all_y_true = []
all_y_pred = []

print(f"Total test images to process: {len(test_image_files)}")

# TODO: Iterate through test_image_files.
# TODO: For each image, run model.predict(), record the inference time, and store predictions.

## 7. Final Quantitative Evaluation & Confusion Matrix
Using the complete set of predictions generated from the Test Set, we calculate the final mAP and Inference Speed (FPS). We also plot the final Confusion Matrix to analyze the true performance and see exactly how many Short, Medium, and Long strawberries were correctly classified.

In [ ]:
# FINAL EVALUATION METRICS AND CONFUSION MATRIX VISUALIZATION FOR ALL MODELS
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 7.1 Plot Final mAP and FPS Comparison
def plot_final_performance_metrics(yolo_metrics, faster_rcnn_metrics, effdet_metrics):
    # TODO: Create bar charts comparing final mAP and FPS across all three models using test set data
    pass

# 7.2 Plot Final Confusion Matrix
def plot_custom_confusion_matrix(y_true, y_pred, classes=['Short', 'Medium', 'Long', 'Background'], title='Final Test Confusion Matrix'):
    """
    Plots the final confusion matrix based on test set predictions and ground truths.
    """
    # TODO: Calculate True Positives, False Positives, False Negatives based on IoU matches

    # Placeholder for visualization structure
    cm_dummy = np.array([[30, 10, 2, 5], [12, 35, 8, 2], [1, 15, 25, 4], [8, 3, 1, 0]])

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_dummy, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Predicted Class')
    plt.ylabel('True Class')
    plt.title(title)
    plt.show()

# TODO: Replace the placeholder variables below with the actual prediction arrays gathered in Step 6
# yolo_true, yolo_pred = ...
# faster_true, faster_pred = ...
# effdet_true, effdet_pred = ...

# Visualize the matrices uniformly for ALL models based on TEST SET results
plot_custom_confusion_matrix(None, None, title="YOLO Final Confusion Matrix")
plot_custom_confusion_matrix(None, None, title="Faster R-CNN Final Confusion Matrix")
plot_custom_confusion_matrix(None, None, title="EfficientDet Final Confusion Matrix")

## 8. Qualitative Visualization (Saving Predicted Test Images)
To visually verify the models' final performance without crashing the notebook output interface, we save all test images with their predicted bounding boxes drawn on them into a specific output directory.

In [ ]:
# VISUALIZE AND SAVE PREDICTED IMAGES
import cv2

# TODO: Define a function to iterate through the test images, draw the predicted bounding boxes, and save them.

def visualize_and_save_predictions(image_paths, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for img_path in image_paths:
        # 1. Load original image
        # 2. Draw bounding boxes (cv2.rectangle) and text (cv2.putText) for the predictions
        # 3. Save to output_dir
        pass

# visualize_and_save_predictions(test_image_files, '/content/drive/MyDrive/strawberry_project/predictions_output/')